### process RAM Tables

In [1]:
from docx import Document
import pandas as pd
import os,sys # ,getpass
sys.path.insert(0,'../libs')
from utils import get_all_files
from utils_pdf import key_reg_match
import re

In [104]:
data_folder = '/root/workspace/data'
word_folder = '/root/workspace/data/ISR_Results/RAM_Tables/by_country_word'
file_paths = get_all_files(word_folder,end_with='.docx')
file_names = [os.path.basename(f) for f in file_paths]


In [105]:
class Word_Table_Extractor():
    def __init__(self, filepath=None):
        self.filepath = filepath
        self.is_valid_table = None
        if filepath:
            self.load_doc(filepath)
        
    def load_doc(self,filepath):
        self.doc = Document(filepath)
        self.tables = self.doc.tables 
        if len(self.tables):
            self.t0 = self.tables[0]
        else:
            self.t0 = None
            
    def clear_doc(self):
        self.doc = None
        self.tables = None
        self.t0 = None
        
    @staticmethod
    def basic_table_extract(table,headers=None):
        data = []
        # Extract column headers
        if headers:
            pass
        else:
            headers = [cell.text.strip() for cell in table.rows[0].cells]

        # Extract row data
        for row in table.rows[1:]:
            row_data = {headers[i]: row.cells[i].text.strip() for i in range(len(row.cells))}  ## there is a problem with using dict type: duplicated headers will be replaced ; fix later 
            data.append(row_data)
            
        return data
    

    

In [106]:
class Word_Table_Identifier(Word_Table_Extractor):
    def __init__(self, filepath=None):
        super().__init__(filepath)

    @staticmethod
    def _remove_text_within_parentheses(text):
        pattern = r"\(.*?\)"
        return re.sub(pattern, '', text).strip().replace("\n"," ") ## also replace ]n
    
    @staticmethod
    def _check_row_column(t0,check_type='row',check_n=3):
        """
        check if there are merged rows or cells 
        """
        ## make sure do not over subscribe 
        if check_type == 'row':
            len_check_item = len(t0.rows)
            check_cells = t0.row_cells
        elif check_type == 'column':
            len_check_item = len(t0.columns)
            check_cells = t0.column_cells
        if not len_check_item>check_n: check_n=len_check_item ## check check to <= max len
        # first filter and get cells that has longer than 7 words 
        ## if string is relatively long, they are not headers ; then they shouldn't have duplicates 
        check_items = [[c.text for c in check_cells(i) if len(c.text.strip().split())>7] for i in range(check_n)]
        for c_i in check_items:
            if len(c_i) != len(set(c_i)):
                return True ## identified dups 
        return False ## no duplicates or merged cells 

    ## rule 1 : check if column headers is valid 
    def _rule_1(self,t0,verbose=True):
        """
        - check first 1-3 row, see if any of the 3 rows roughly contains risks; likelihood
        - raw info shouldn't be too long (after removing info in "()"); sometime they can have something like this "Source of Risks and Relative Likelihood (High, medium, or low)"
        """
        if len(t0.rows)<=2:
            if verbose:
                print('rule 1 failed: table 0 less than 3 rows')
            return False
        
        for row in t0.rows[:3]:
            maybe_headers_strings = " ".join([cell.text.strip() for cell in row.cells])
            maybe_headers_strings = self._remove_text_within_parentheses(maybe_headers_strings)
            pattern = re.compile(r'(?=.*(?:risk|threat|shock|source))(?=.*(?:likelihoo|impact))', re.I) 
            res = key_reg_match(maybe_headers_strings,reg_text = pattern,mode='rgx')
            if res:
                if len(maybe_headers_strings.split()) < 30:
                    return True ## columns header identified ; pass
        if verbose:
            print('Rule 1 failed: no header identified')
        return False ## no header fund 
    
    ## rule 2 : check if if there are merged columns ; by checking if there are duplicated cells in a row ;  
    def _rule_2(self,t0,verbose=True):
        if self._check_row_column(t0,check_type='row',check_n=7): ## if identified dups
            if verbose:
                print('rule 2 failed: identified merged cells row wise... ')
            return False 
        
        if self._check_row_column(t0,check_type='column',check_n=2): ## if identified dups
            if verbose:
                print('rule 2 failed: identified merged cells column wise ... ')
            return False 
        
        return True # if no dup found, return True 
    
    ## rule 3 : sometimes, the entire row or table merged into once cell check for extremely long cell content or check for multiple \n\n\n
    def _rule_3(self,t0,verbose=True):
        ## check all cells ; shouldn't have any cell value that is strange 
        for row in t0.rows:
            for c in row.cells:
                text = c.text
                triple_newline_occurrences = text.count("\n\n\n") > 2
                # Check if the number of words is over 500
                word_count_over= len(text.split()) > 200
                if  triple_newline_occurrences or word_count_over:
                    if verbose:
                        print('rule 3 failed: large merged cell identified')
                    return False ## identified problem  cell value
        
        return True
    
    def identify_non_standard_tables(self,table=None,doc_path=None,verbose=True):
        if doc_path:
            self.load_doc(doc_path)
            if len(self.tables) == 0:
                if verbose:
                    print('rule 0 failed: no table found')
                return False
            
        if table:
            self.clear_doc()
            self.t0 = table
            
        rules = [self._rule_1,self._rule_2,self._rule_3]
        for r in rules:
            if not r(self.t0,verbose):
                return False
        
        return True

### Things to add 
- if first column and second is exact the same. first col can be ignored. (for now we don't treat them as standard table;maybe want to change later)
- if first column and second is exact the same, but have duplicate column wise ; first colum is column meta info 

In [107]:
R = Word_Table_Identifier()

In [108]:
for fp in  file_paths:
    if not R.identify_non_standard_tables(doc_path = fp):
        print(fp)

rule 0 failed: no table found
/root/workspace/data/ISR_Results/RAM_Tables/by_country_word/138_2022_1.docx
Rule 1 failed: no header identified
/root/workspace/data/ISR_Results/RAM_Tables/by_country_word/278_2022_0.docx
rule 2 failed: identified merged cells row wise... 
/root/workspace/data/ISR_Results/RAM_Tables/by_country_word/298_2023_0.docx
rule 2 failed: identified merged cells column wise ... 
/root/workspace/data/ISR_Results/RAM_Tables/by_country_word/449_2022_0.docx
rule 2 failed: identified merged cells column wise ... 
/root/workspace/data/ISR_Results/RAM_Tables/by_country_word/466_2022_0.docx
rule 2 failed: identified merged cells row wise... 
/root/workspace/data/ISR_Results/RAM_Tables/by_country_word/536_2023_0.docx
rule 3 failed: large merged cell identified
/root/workspace/data/ISR_Results/RAM_Tables/by_country_word/566_2022_0.docx
rule 2 failed: identified merged cells row wise... 
/root/workspace/data/ISR_Results/RAM_Tables/by_country_word/672_2022_0.docx
rule 2 failed:

- basic debugging

In [109]:
fp = '/root/workspace/data/ISR_Results/RAM_Tables/by_country_word/672_2022_0.docx'
R.load_doc(fp)
check_items = [[c.text for c in R.t0.row_cells(i)] for i in range(6)]
check_items

[['Sources',
  'Sources',
  'Likelihood',
  'Potential Impact',
  'Policies to Minimize Impact'],
 ['External', 'External', 'External', 'External', 'External'],
 ['Rising and volatile food and energy prices.',
  'Rising and volatile food and energy prices.',
  'High',
  'Medium. Rising gas and rice prices reduce households’ purchasing power and push up poverty. The higher import bill could become a drain on international reserves unless prices for iron ore and gold exports rise commensurately.',
  'Generally, allow pass-through of import prices to domestic prices. Any relief measures for the population should be targeted to the extent possible, cost effective, transparent, and affordable.'],
 ['Outbreaks of lethal and highly contagious COVID-19 variants lead to subpar/volatile growth, with increased divergence across countries.',
  'Outbreaks of lethal and highly contagious COVID-19 variants lead to subpar/volatile growth, with increased divergence across countries.',
  'Medium',
  'Hi

In [3]:
import pandas as pd 
from utils import load_json, extract_json_string,get_all_files
from oai_ast_utils import OpenAIAssistant_Base
import tqdm,json,time
#from openai import OpenAI
## load API Key
key = load_json('/root/workspace/key/openai_key.json') 
os.environ['OPENAI_API_KEY'] = key['ISR']['API_KEY']

- Upload Files first 

In [10]:
RAM_Bot = OpenAIAssistant_Base()
RAM_Bot.FileManager._get_file_info()

,id,bytes,created_at,filename,object,purpose,status,status_details
0,file-Yw7nmrMzFSMbHWiiBsjyh1yF,157390,1706763077,test.pdf,file,assistants,processed,None


In [29]:
pdf_folder = '/root/workspace/data/DOCs/RAM/PDF'
all_pdfs = [f for f in get_all_files(pdf_folder,end_with='.pdf') if '_RAM' in f]
file_names = [os.path.basename(f) for f in all_pdfs]

In [25]:
UPLOAD_FILES = False
DELETE_FILES = False
CREATE_AST = False

In [22]:
if UPLOAD_FILES:
    
    if DELETE_FILES:
    # clear all previous files 
        filter_criteria={} #USA_2022.pdf
        sampled_file_ids = RAM_Bot.FileManager.get_files_info_by(filter_criteria,return_fields=['id'],to_dict=True,to_single_list=True)
        RAM_Bot.FileManager.delete_files_by_ids(file_ids=sampled_file_ids)
        print('files deleted {}'.format(sampled_file_ids))
    
    ## upload files get files ids
    files_list = []
    for fp in all_pdfs:
        fid,fname = RAM_Bot.FileManager.upload_file(fp,purpose='assistants')
        files_list.append((fid,fname))
        
    print(RAM_Bot.FileManager._get_file_info())

['file-Yw7nmrMzFSMbHWiiBsjyh1yF'] has been removed from file server.
files deleted ['file-Yw7nmrMzFSMbHWiiBsjyh1yF']
                              id   bytes  created_at                filename  \
0  file-Votz199fcYFeUlSn7DrplxDx  157390  1706764100  Indonesia_2023_RAM.pdf   
1  file-CK5vhwJd0gRukzOaSqy4vCE5   74304  1706764100        UAE_2021_RAM.pdf   

  object     purpose     status status_details  
0   file  assistants  processed           None  
1   file  assistants  processed           None  


In [26]:
if CREATE_AST: 
    RAM_Bot.create_assistant(name='PDF_Table_Extractor',
                        description='This bot is created to help PDF Table Extraction',
                        instructions="You are an agent to extract and clean messy data extracted from PDFs. You should use your best judgment to clean the data and format them into a clean format as user instructed .")
    print(RAM_Bot.assistant.name , RAM_Bot.assistant.id)

In [24]:
## reload assistant ; sometime there are bugs if keep reusing the same assistant
filter_criteria={'name':['PDF_Table_Extractor']}
as_id = RAM_Bot.get_assistants_info_by(filter_criteria,return_fields=['id'])
print(as_id)
## retrieve by id 
assit = RAM_Bot.get_assistants_by_ids(as_id[0])
print(assit.name, assit.id )
## set the retrieved assistant as current active assistant 
RAM_Bot._set_active_assistant(assit)

['asst_cevVRdGRRRvz06V2Gu0ZoJkA']
PDF_Table_Extractor asst_cevVRdGRRRvz06V2Gu0ZoJkA
set PDF_Table_Extractor as current active assistant.


In [27]:
RAM_Bot.FileManager._get_file_info()

,id,bytes,created_at,filename,object,purpose,status,status_details
0,file-Votz199fcYFeUlSn7DrplxDx,157390,1706764100,Indonesia_2023_RAM.pdf,file,assistants,processed,None
1,file-CK5vhwJd0gRukzOaSqy4vCE5,74304,1706764100,UAE_2021_RAM.pdf,file,assistants,processed,None


In [31]:
print(file_names)

['Indonesia_2023_RAM.pdf', 'UAE_2021_RAM.pdf']


In [32]:
loop_files = RAM_Bot.FileManager.get_files_info_by(filter_criteria={'filename':file_names},return_fields=['id','filename'],to_dict=True)
print(loop_files)

[{'id': 'file-Votz199fcYFeUlSn7DrplxDx', 'filename': 'Indonesia_2023_RAM.pdf'}, {'id': 'file-CK5vhwJd0gRukzOaSqy4vCE5', 'filename': 'UAE_2021_RAM.pdf'}]


In [33]:
user_prompt = """ 
Please extract information from the table row by row. Please return a clean list. each item of the list and have clean information under each column name; for example:

1)
Source of Risks: ...... || Relative Likelihood: .... || Expected Impact : .... || Policy Response : ....
2)
Source of Risks: ...... || Relative Likelihood: .... || Expected Impact : .... || Policy Response : ....
3) 
... ...


Please make sure you directly copy content from the sources as much as possible. Do not summarize raw information.
Please proceed:
"""

In [34]:
results = {}
for f_dict in tqdm.tqdm(loop_files):
## update assistant file with specific country file 
    fid,fname = f_dict.get('id'),f_dict.get('filename')
    ass_info_dict = {
                'model':"gpt-4-1106-preview",
                'file_ids':[fid] ### USA file id
                }
    RAM_Bot.update_current_assistant(**ass_info_dict)
    
    user_input_dict = {"role":"user",
                    "content":user_prompt
                    }
    msg,status = RAM_Bot.quick_query(user_input_dict)
    results[fname]={'msg':msg,'status':status}
    time.sleep(2)

  0%|          | 0/2 [00:00<?, ?it/s]

Elapsed time: 37.92 s || Status: completed      

 50%|█████     | 1/2 [00:41<00:41, 41.38s/it]

Elapsed time: 48.80 s || Status: completed      

100%|██████████| 2/2 [01:33<00:00, 46.78s/it]


In [41]:
results.keys()

dict_keys(['Indonesia_2023_RAM.pdf', 'UAE_2021_RAM.pdf'])

In [42]:
print(results['UAE_2021_RAM.pdf']['msg'])

The document provides a Risk Assessment Matrix (RAM) from the IMF staff regarding various global risks and their impacts on the economy, along with the corresponding policy responses. Here is the clean list extracted from the table:

1)
Source of Risks: Global resurgence of the COVID-19 pandemic.
|| Relative Likelihood: High/Short-to-medium-term
|| Expected Impact: Renewed containment measures and travel restrictions would severely impact hard-hit sectors reducing non-oil growth. Declining global demand would lead to oil price declines eroding fiscal and external balances through lower oil revenue and possibly also result in additional OPEC+ production cuts and lower oil GDP growth. Global financial conditions would tighten increasing risks of capital flows reversals and vulnerabilities among highly indebted GREs.
|| Policy Response: Health measures should be strengthened as needed. Targeted fiscal and macro-financial support measures should be extended. Additional fiscal space should 

In [40]:
res_folder = '/root/workspace/data/DOCs/RAM'
with open(os.path.join(res_folder,'raw_table_extraction_msg.json'), 'w') as file:
    json.dump(results, file, indent=4)